In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")
from statsmodels.graphics.tsaplots import plot_acf

from src.database.connection import get_connection

import warnings
warnings.filterwarnings(
    "ignore",
    message = "pandas only supports SQLAlchemy connectable.*",
    category = UserWarning
)

In [ ]:
conn = get_connection()

query = """
    SELECT *
    FROM generation_eda
    ORDER BY start_time
"""

generation = pd.read_sql(query, conn)
generation["publish_time"] = pd.to_datetime(generation["publish_time"], utc = True)
generation["start_time"] = pd.to_datetime(generation["start_time"], utc = True)

query = """
    SELECT 
        *
    FROM demand_eda;
"""

demand = pd.read_sql(query, conn)
demand["start_time"] = pd.to_datetime(demand["start_time"], utc = True)

conn.close()

generation.head()

In [ ]:
generation.describe()

In [ ]:
generation.groupby("fuel_type")["start_time"].diff().value_counts().sort_index()

In [ ]:
generation.groupby("fuel_type").agg(rows = ("generation_mw", "size"))

In [ ]:
diffs = generation.groupby("fuel_type")["start_time"].diff()
generation.loc[diffs == pd.Timedelta("1 hour")]

In [ ]:
generation.head()

In [ ]:
fill_generation = generation.copy()

fill_generation = generation.sort_values(["fuel_type", "start_time"])
fill_generation["previous_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(1)
fill_generation["next_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(-1)
fill_generation["previous_start_time"] = fill_generation.groupby("fuel_type")["start_time"].shift(1)
fill_generation["interval"] = fill_generation["start_time"] - fill_generation["previous_start_time"]

wrong_intervals = fill_generation[fill_generation["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["publish_time"] = wrong_intervals["start_time"]
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["generation_mw"] = ((
    wrong_intervals["previous_generation"] + wrong_intervals["next_generation"]
    ) / 2).round(0)

wrong_intervals = wrong_intervals[["publish_time", "start_time", "fuel_type", "generation_mw"]]

generation = pd.concat([generation, wrong_intervals], ignore_index = True).sort_values("start_time")

In [ ]:
fill_demand = demand.copy()

fill_demand["previous_demand"] = fill_demand["true_demand_mw"].shift(1)
fill_demand["next_demand"] = fill_demand["true_demand_mw"].shift(-1)
fill_demand["previous_start_time"] = fill_demand["start_time"].shift(1)
fill_demand["interval"] = fill_demand["start_time"] - fill_demand["previous_start_time"]

wrong_intervals = fill_demand[fill_demand["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["true_demand_mw"] = ((
    wrong_intervals["previous_demand"] + wrong_intervals["next_demand"]
    ) / 2).round(0)

wrong_intervals = wrong_intervals[["start_time", "true_demand_mw"]]

demand = pd.concat([demand, wrong_intervals], ignore_index = True).sort_values("start_time")
demand = demand.rename(columns = {"start_time": "prediction_time"})

In [ ]:
generation_pivot = generation.pivot_table(
    index = "publish_time",
    columns = "fuel_type",
    values = "generation_mw",
    aggfunc = "last"
).reset_index().sort_values("publish_time")

generation_pivot.head()

In [ ]:
modelling = pd.merge_asof(
    left = demand,
    right = generation_pivot,
    left_on = "prediction_time",
    right_on = "publish_time",
    direction = "backward"
)

modelling.head()

In [ ]:
modelling = modelling.drop(index = 0).reset_index(drop = True)

modelling["interval"] = modelling["prediction_time"] - modelling["publish_time"]
modelling["interval"].value_counts()

In [ ]:
wrong_intervals = modelling[modelling["interval"] > pd.Timedelta(0)]
wrong_intervals[["prediction_time", "publish_time", "interval"]].head(20)

In [ ]:
#https://bscdocs.elexon.co.uk/interface-definition-documents/neta-interface-definition-and-design-document-part-1-interfaces-with-bsc-parties-and-their-agents?utm_source=chatgpt.com
#4.13.4.42 Fuel Type

fuels = [
    "BIOMASS",
    "WIND",
    "PS",
    "OTHER",
    "OIL",
    "OCGT",
    "NPSHYD",
    "NUCLEAR",
    "COAL",
    "CCGT"
]

interconnectors = [
    "INTNSL",
    "INTNEM",
    "INTIRL",
    "INTIFA2",
    "INTFR",
    "INTEW",
    "INTELEC",
    "INTNED"
]

In [ ]:
modelling[fuels].describe().T

In [ ]:
modelling[interconnectors].describe().T

In [ ]:
fig, axes = plt.subplots(2, 5, figsize = (20, 6), constrained_layout = True)
axes = axes.flatten()

for fuel, ax in zip(fuels, axes):
    sns.histplot(
        data = modelling,
        x = fuel,
        bins = 50,
        ax = ax
    )

    ax.set_title(f"{fuel} Generation Distribution")
    ax.set_xlabel("Generation (MW)")
    ax.set_ylabel("Frequency")

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (16, 6), constrained_layout = True)
axes = axes.flatten()

for interconnector, ax in zip(interconnectors, axes):
    sns.histplot(
        data = modelling,
        x = fuel,
        bins = 50,
        ax = ax
    )

    ax.set_title(f"{interconnector} Generation Distribution")
    ax.set_xlabel("Generation (MW)")
    ax.set_ylabel("Frequency")

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize = (20, 6), constrained_layout = True)
axes = axes.flatten()

for fuel, ax in zip(fuels, axes):
    sns.boxplot(
        data = modelling,
        x = fuel,
        ax = ax
    )

    ax.set_title(f"{fuel} Generation Distribution")
    ax.set_xlabel("Generation (MW)")
    ax.set_ylabel("Frequency")

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (16, 6), constrained_layout = True)
axes = axes.flatten()

for interconnector, ax in zip(interconnectors, axes):
    sns.boxplot(
        data = modelling,
        x = fuel,
        ax = ax
    )

    ax.set_title(f"{interconnector} Generation Distribution")
    ax.set_xlabel("Generation (MW)")
    ax.set_ylabel("Frequency")

plt.show()

In [ ]:
total_rows = modelling.shape[0]
no_oil = 100.0 * modelling[modelling["OIL"] == 0].shape[0] / total_rows
no_ocgt = 100.0 * modelling[modelling["OCGT"] == 0].shape[0] / total_rows
no_coal = 100.0 * modelling[modelling["COAL"] == 0].shape[0] / total_rows

print(f"NO OIL: {no_oil:.2f}%\nNO OCGT: {no_ocgt:.2f}%\nNO COAL: {no_coal:.2f}%")

In [ ]:
fuel_correlation = modelling[["true_demand_mw"] + fuels].corr()

plt.figure(figsize = (8, 6), constrained_layout = True)

sns.heatmap(
    data = fuel_correlation,
    vmin = -1,
    cmap = "coolwarm",
    annot = True,
    fmt = ".2f"
)

plt.title("Fuel Correlations")

plt.show()

In [ ]:
interconnector_correlation = modelling[["true_demand_mw"] + interconnectors].corr()

plt.figure(figsize = (7, 5), constrained_layout = True)

sns.heatmap(
    data = interconnector_correlation,
    vmin = -1,
    cmap = "coolwarm",
    annot = True,
    fmt = ".2f"
)

plt.title("Interconnector Correlations")

plt.show()

In [ ]:
modelling["fuels_total"] = modelling[fuels].sum(axis = 1)
modelling["interconnectors_total"] = modelling[interconnectors].sum(axis = 1)
modelling["total_generation"] = modelling["fuels_total"] + modelling["interconnectors_total"]
to_plot = {
    "fuels_total": "Fuel",
    "interconnectors_total": "Interconnector",
    "total_generation": "Total"
}

daily_model = modelling.set_index("prediction_time").resample("D").agg({
    "true_demand_mw": "mean",
    "fuels_total": "mean",
    "interconnectors_total": "mean",
    "total_generation": "mean"
}).reset_index()

fig, axes = plt.subplots(1, 3, figsize = (10, 4), constrained_layout = True)
axes = axes.flatten()

for (row, label), ax in zip(to_plot.items(), axes):
    sns.scatterplot(
        data = daily_model,
        x = row,
        y = "true_demand_mw",
        ax = ax
    )

    ax.set_title(f"Monthly Demand vs {label} Generation")
    ax.set_ylabel("")
    ax.set_xlabel(f"Monthly {label} Generation (MW)")

axes[0].set_ylabel("Monthly True Demand (MW)")

plt.show()

In [ ]:
sums_correlation = modelling[["true_demand_mw"] + list(to_plot.keys())].corr()

plt.figure(figsize = (5, 4), constrained_layout = True)

sns.heatmap(
    data = sums_correlation,
    vmin = -1,
    cmap = "coolwarm",
    annot = True,
    fmt = ".2f"
)

plt.title("Sum Correlations")

plt.show()